In [1]:
%pip install pandas ta

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29482 sha256=44e43a53d4a75551a0c27c1db754072c8c87eec4dfcf707d99c56aba8d43364b
  Stored in directory: /Users/melaniequ/Library/Caches/pip/wheels/e3/3a/ee/4955a26c90a4b7deb6d725dc8ec7b8604a7aef44e43a2e8af7
Successfully built ta
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

from ta.trend import (
    SMAIndicator,
    EMAIndicator,
    MACD
)

from ta.momentum import RSIIndicator

from ta.volatility import AverageTrueRange


def calculate_indicators(df):
    """
    Calculate technical indicators for BTC market data.

    Required columns:
        timestamp
        open
        high
        low
        close
        volume

    Returns:
        pandas.DataFrame
    """

    # Make a copy so we don't modify the original DataFrame
    df = df.copy()

    # ---------------------------------------------------------
    # Simple Moving Averages
    # ---------------------------------------------------------

    df["sma_20"] = SMAIndicator(
        close=df["close"],
        window=20
    ).sma_indicator()

    df["sma_50"] = SMAIndicator(
        close=df["close"],
        window=50
    ).sma_indicator()

    # ---------------------------------------------------------
    # Exponential Moving Averages
    # ---------------------------------------------------------

    df["ema_12"] = EMAIndicator(
        close=df["close"],
        window=12
    ).ema_indicator()

    df["ema_26"] = EMAIndicator(
        close=df["close"],
        window=26
    ).ema_indicator()

    # ---------------------------------------------------------
    # RSI
    # ---------------------------------------------------------

    df["rsi_14"] = RSIIndicator(
        close=df["close"],
        window=14
    ).rsi()

    # ---------------------------------------------------------
    # MACD
    # ---------------------------------------------------------

    macd = MACD(
        close=df["close"],
        window_fast=12,
        window_slow=26,
        window_sign=9
    )

    df["macd"] = macd.macd()
    df["macd_signal"] = macd.macd_signal()
    df["macd_histogram"] = macd.macd_diff()

    # ---------------------------------------------------------
    # ATR
    # ---------------------------------------------------------

    atr = AverageTrueRange(
        high=df["high"],
        low=df["low"],
        close=df["close"],
        window=14
    )

    df["atr_14"] = atr.average_true_range()

    # ---------------------------------------------------------
    # Volume indicators
    # ---------------------------------------------------------

    df["volume_sma_20"] = (
        df["volume"]
        .rolling(window=20)
        .mean()
    )

    df["volume_ratio"] = (
        df["volume"] / df["volume_sma_20"]
    )

    # ---------------------------------------------------------
    # Price change
    # ---------------------------------------------------------

    df["price_change_pct"] = (
        df["close"]
        .pct_change() * 100
    )

    # ---------------------------------------------------------
    # Remove rows where indicators cannot yet be calculated
    # ---------------------------------------------------------

    df = df.dropna().reset_index(drop=True)

    return df

In [7]:
import pandas as pd

df = pd.read_csv("../data/btc_candles.csv")
df_indicators = calculate_indicators(df)

print(df_indicators.columns.tolist())

['timestamp', 'open', 'high', 'low', 'close', 'volume', 'sma_20', 'sma_50', 'ema_12', 'ema_26', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'atr_14', 'volume_sma_20', 'volume_ratio', 'price_change_pct']


In [8]:
df_indicators[
    [
        "timestamp",
        "close",
        "sma_20",
        "sma_50",
        "ema_12",
        "ema_26",
        "rsi_14",
        "macd",
        "macd_signal",
        "atr_14",
        "volume_ratio"
    ]
].tail()

,timestamp,close,sma_20,sma_50,ema_12,ema_26,rsi_14,macd,macd_signal,atr_14,volume_ratio
90,2026-08-31 02:30:00+00:00,77457.35,78354.2810,78362.6408,77964.681987,78219.681333,32.108536,-254.999346,-146.503572,386.416376,0.685910
91,2026-08-31 03:00:00+00:00,77708.57,78293.4470,78356.1928,77925.280143,78181.821234,39.202720,-256.541091,-168.511076,385.515920,0.642342
92,2026-08-31 03:30:00+00:00,77744.84,78230.1015,78349.1308,77897.520121,78149.452254,40.174684,-251.932133,-185.195287,372.036212,0.712958
93,2026-08-31 04:00:00+00:00,77500.01,78155.1120,78335.7394,77836.364718,78101.345420,35.991838,-264.980703,-201.152370,369.207911,0.700649
94,2026-08-31 04:30:00+00:00,77524.59,78079.9385,78323.5292,77788.399377,78058.622797,36.704355,-270.223420,-214.966580,352.515203,0.294049


In [9]:
latest = df_indicators.iloc[-1]

print("BTC Price:", latest["close"])
print("ATR(14):", latest["atr_14"])

BTC Price: 77524.59
ATR(14): 352.51520307606495


In [10]:
atr_multiplier = 1.5

stop_price = (
    latest["close"]
    - atr_multiplier * latest["atr_14"]
)

print("Hypothetical ATR stop:", stop_price)

Hypothetical ATR stop: 76995.8171953859


In [11]:
df_indicators.to_csv(
    "../data/btc_indicators.csv",
    index=False
)

print("Indicator data saved.")

Indicator data saved.
